In [5]:
import os
import shutil
import random
import xml.etree.ElementTree as ET
from tqdm import tqdm
from PIL import Image
import glob

# ================= 配置区域 =================
# 1. 原始数据根目录 (包含文件夹 1-10 和 label)
raw_data_root = "/root/autodl-tmp/GC10-DET" 

# 2. 输出路径
output_root = "/root/autodl-tmp/exp1/GC10_DET_YOLO"

# 3. 标签 XML 所在文件夹
label_folder_path = os.path.join(raw_data_root, "label")

# ================= 核心逻辑：基于文件夹定类别 =================
def convert_box(size, box):
    dw = 1. / size[0]
    dh = 1. / size[1]
    x = (box[0] + box[1]) / 2.0
    y = (box[2] + box[3]) / 2.0
    w = box[1] - box[0]
    h = box[3] - box[2]
    return (x * dw, y * dh, w * dw, h * dh)

def process_dataset():
    # 1. 准备目录
    if os.path.exists(output_root): shutil.rmtree(output_root)
    for split in ['train', 'val']:
        os.makedirs(os.path.join(output_root, 'images', split), exist_ok=True)
        os.makedirs(os.path.join(output_root, 'labels', split), exist_ok=True)

    print(f"🚀 开始基于文件夹结构的强制转换...")
    print(f"📂 图片源: {raw_data_root}/1-10")
    print(f"📂 标签源: {label_folder_path}")

    # 2. 定义映射 (文件夹名 -> YOLO ID)
    # 文件夹 '1' -> ID 0 (chongkong)
    # 文件夹 '10' -> ID 9 (yaozhe)
    folder_to_id = {str(i): i-1 for i in range(1, 11)}
    
    # 收集所有数据任务
    # 格式: (图片路径, XML路径, 强制类别ID)
    tasks = []

    for folder_name, class_id in folder_to_id.items():
        folder_path = os.path.join(raw_data_root, folder_name)
        if not os.path.exists(folder_path):
            print(f"⚠️ 警告: 文件夹 {folder_name} 不存在，跳过。")
            continue
            
        # 扫描该文件夹下的图片
        images = glob.glob(os.path.join(folder_path, "*.[jJ][pP][gG]"))
        
        for img_path in images:
            file_name = os.path.basename(img_path)
            base_name = os.path.splitext(file_name)[0]
            
            # 找 XML (只负责找坐标)
            xml_path = os.path.join(label_folder_path, base_name + '.xml')
            if not os.path.exists(xml_path):
                xml_path = os.path.join(label_folder_path, base_name + '.XML')
            
            if os.path.exists(xml_path):
                tasks.append((img_path, xml_path, class_id))
            # else: print(f"缺失 XML: {base_name}")

    print(f"📊 共收集到 {len(tasks)} 个有效图文对。正在划分并转换...")
    
    # 3. 划分数据集
    random.shuffle(tasks)
    split_idx = int(len(tasks) * 0.9)
    train_tasks = tasks[:split_idx]
    val_tasks = tasks[split_idx:]
    
    # 4. 执行转换
    success_count = 0
    
    def run_convert(task_list, split_name):
        nonlocal success_count
        for img_path, xml_path, target_cls_id in tqdm(task_list, desc=f"Processing {split_name}"):
            try:
                # 解析 XML 获取尺寸和坐标
                tree = ET.parse(xml_path)
                root = tree.getroot()
                
                size = root.find('size')
                if size is not None:
                    w = int(size.find('width').text)
                    h = int(size.find('height').text)
                else:
                    img = Image.open(img_path)
                    w, h = img.size
                
                if w == 0 or h == 0:
                    img = Image.open(img_path)
                    w, h = img.size

                label_txt = ""
                has_obj = False
                
                # 遍历所有 object，提取坐标，但无视 XML 里的 name
                # 直接使用 target_cls_id
                for obj in root.iter('object'):
                    xmlbox = obj.find('bndbox')
                    b = (float(xmlbox.find('xmin').text), float(xmlbox.find('xmax').text), 
                         float(xmlbox.find('ymin').text), float(xmlbox.find('ymax').text))
                    
                    # 转换坐标
                    bb = convert_box((w, h), b)
                    
                    # 🔥 强制写入文件夹对应的类别 ID
                    label_txt += f"{target_cls_id} {bb[0]:.6f} {bb[1]:.6f} {bb[2]:.6f} {bb[3]:.6f}\n"
                    has_obj = True
                
                if has_obj:
                    file_name = os.path.basename(img_path)
                    base_name = os.path.splitext(file_name)[0]
                    
                    # 复制图片
                    shutil.copy(img_path, os.path.join(output_root, 'images', split_name, file_name))
                    # 写入标签
                    with open(os.path.join(output_root, 'labels', split_name, base_name + ".txt"), 'w') as f:
                        f.write(label_txt)
                    success_count += 1
                    
            except Exception as e:
                pass

    run_convert(train_tasks, 'train')
    run_convert(val_tasks, 'val')
    
    print(f"\n✅ 转换完成！共生成 {success_count} 个样本。")
    print(f"   文件夹 1 -> Class 0 (冲孔)")
    print(f"   文件夹 2 -> Class 1 (焊缝)")
    print(f"   ...")
    print(f"   文件夹 10 -> Class 9 (腰折)")

if __name__ == "__main__":
    process_dataset()

🚀 开始基于文件夹结构的强制转换...
📂 图片源: /root/autodl-tmp/GC10-DET/1-10
📂 标签源: /root/autodl-tmp/GC10-DET/label
📊 共收集到 2306 个有效图文对。正在划分并转换...


Processing val: 100%|██████████| 231/231 [00:00<00:00, 395.21it/s]


✅ 转换完成！共生成 2304 个样本。
   文件夹 1 -> Class 0 (冲孔)
   文件夹 2 -> Class 1 (焊缝)
   ...
   文件夹 10 -> Class 9 (腰折)


In [1]:
import os
from ultralytics import YOLO

# 数据集配置文件路径
DATA_YAML = "/root/autodl-tmp/exp1/GC10_DET_YOLO/gc10.yaml"

print("🚀 开始 Exp 13: GC10-DET 大规模源域预训练...")

model = YOLO('yolo11n.pt') # 依然从 ImageNet 开始

model.train(
    data=DATA_YAML,
    epochs=100,      # 建议跑久一点，作为基础模型越强越好
    batch=32,        # 如果显存够大，可以开 64
    imgsz=640,
    project='result_exp1',
    name='13_GC10_Pretrain_Base',
    device='0',
    exist_ok=True,
    patience=15,     # 如果15轮不涨分就停
    close_mosaic=10, # 最后10轮关闭马赛克增强，精细化训练
    verbose=True
)

🚀 开始 Exp 13: GC10-DET 大规模源域预训练...
New https://pypi.org/project/ultralytics/8.3.252 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.248 🚀 Python-3.10.8 torch-2.1.2+cu118 CUDA:0 (NVIDIA GeForce RTX 4090, 24111MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/root/autodl-tmp/exp1/GC10_DET_YOLO/gc10.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7fe7c4bc6590>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.0

In [2]:
import os
from ultralytics import YOLO
from ultralytics.models.yolo.detect import DetectionTrainer
import torch
import torch.nn as nn
import numpy as np

# ================= 1. 隐私引擎 (复用 Exp 11 的最强版: SA-LDP + AGC) =================
class PrivacyEngine_Transfer:
    def __init__(self, model, epsilon=10.0, strategy='adaptive_smart'):
        self.model = model
        self.epsilon = epsilon
        self.strategy = strategy
        
        # 自动寻找 Head
        self.head_indices = self._find_head_indices()
        self.backbone_indices = list(range(10))

    def _find_head_indices(self):
        indices = []
        for name, module in self.model.named_modules():
            if hasattr(module, '__class__') and 'Detect' in str(module.__class__):
                try:
                    idx = int(name.split('.')[1])
                    indices.append(idx)
                except: pass
        if not indices: indices = [23]
        return indices

    def step(self, current_epoch):
        if current_epoch < 3: return # Warmup

        # AGC 动态裁剪
        grad_norms = []
        for p in self.model.parameters():
            if p.requires_grad and p.grad is not None:
                grad_norms.append(p.grad.norm(2).item())
        
        if not grad_norms: return
        current_median = np.median(grad_norms)
        clip_val = max(0.1, min(current_median * 1.5, 10.0))

        # 策略分配
        current_device = next(self.model.parameters()).device
        names_list = [n for n, p in self.model.named_parameters() if p.requires_grad]
        factors = []
        
        for n in names_list:
            layer_factor = 1.0
            try:
                layer_idx = int(n.split('.')[1])
                if self.strategy == 'adaptive_smart':
                    if layer_idx in self.head_indices: layer_factor = 1.5
                    elif layer_idx in self.backbone_indices: layer_factor = 0.8
            except: pass
            factors.append(layer_factor)

        factors = torch.tensor(factors, device=current_device)
        weights = factors / factors.sum() * len(factors) 
        
        # 加噪
        idx = 0
        for p in self.model.parameters():
            if p.requires_grad and p.grad is not None:
                layer_eps = self.epsilon * weights[idx]
                torch.nn.utils.clip_grad_norm_(p, clip_val)
                c = np.sqrt(2 * np.log(1.25 / 1e-5))
                sigma = c * clip_val / (layer_eps + 1e-8)
                p.grad.add_(torch.randn_like(p.grad) * sigma)
                idx += 1

# ================= 2. 训练器 =================
class Trainer_Transfer(DetectionTrainer):
    def get_model(self, cfg=None, weights=None, verbose=True):
        # 注意：这里我们加载的是 GC10 预训练权重
        model = super().get_model(cfg, weights, verbose)
        self.privacy_engine = PrivacyEngine_Transfer(model, epsilon=10.0, strategy='adaptive_smart')
        return model

    def optimizer_step(self):
        self.scaler.unscale_(self.optimizer)
        self.privacy_engine.step(self.epoch)
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), 10.0)
        self.scaler.step(self.optimizer)
        self.scaler.update()
        self.optimizer.zero_grad()
        if self.ema: self.ema.update(self.model)

# ================= 3. 执行 Exp 14 (迁移验证) =================
# 你的 NEU-DET 配置
NEU_YAML = "/root/autodl-tmp/exp1/NEU_DET_YOLO/neu_det.yaml"

# 🔥 关键：刚才 Exp 13 跑出来的最佳权重
# 请检查路径是否正确，通常是 weights/best.pt
PRETRAINED_GC10 = "/root/autodl-tmp/exp1/result_exp1/13_GC10_Pretrain_Base/weights/best.pt"

print(f"🚀 开始 Exp 14: 从 GC10 (mAP=0.58) 迁移到 NEU-DET...")
print(f"📦 加载权重: {PRETRAINED_GC10}")

if os.path.exists(PRETRAINED_GC10):
    trainer = Trainer_Transfer(overrides={
        'model': PRETRAINED_GC10,  # 👈 使用 GC10 权重
        'data': NEU_YAML,          # 👈 训练 NEU 数据
        'epochs': 30,
        'batch': 16,
        'imgsz': 640,
        'project': 'result_exp1',
        'name': '14_Transfer_GC10_SALDP',
        'device': '0',
        'exist_ok': True,
        'freeze': 10  # ❄️ 冻结 Backbone！相信 GC10 学到的纹理特征！
    })
    trainer.train()
else:
    print(f"❌ 找不到权重文件: {PRETRAINED_GC10}")

🚀 开始 Exp 14: 从 GC10 (mAP=0.58) 迁移到 NEU-DET...
📦 加载权重: /root/autodl-tmp/exp1/result_exp1/13_GC10_Pretrain_Base/weights/best.pt
Ultralytics 8.3.248 🚀 Python-3.10.8 torch-2.1.2+cu118 CUDA:0 (NVIDIA GeForce RTX 4090, 24111MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/root/autodl-tmp/exp1/NEU_DET_YOLO/neu_det.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=10, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/root/autodl-tmp/exp1/result_exp1/13_GC10_Pretrai

In [1]:
import os
from ultralytics import YOLO
from ultralytics.models.yolo.detect import DetectionTrainer
import torch
import torch.nn as nn
import numpy as np
import math

# ================= 1. 噪声退火引擎 (复用 Exp 16 逻辑) =================
class PrivacyEngine_Annealing_Rescue:
    def __init__(self, model, epsilon=10.0, strategy='uniform', epochs=30):
        self.model = model
        self.epsilon = epsilon
        self.strategy = strategy # 这里我们将传入 'uniform'
        self.total_epochs = epochs
        
        # 依然需要识别 Head，但如果是 uniform 策略，这个其实不影响计算
        self.head_indices = self._find_head_indices()
        self.backbone_indices = list(range(10))

    def _find_head_indices(self):
        indices = []
        for name, module in self.model.named_modules():
            if hasattr(module, '__class__') and 'Detect' in str(module.__class__):
                try:
                    idx = int(name.split('.')[1])
                    indices.append(idx)
                except: pass
        if not indices: indices = [23]
        return indices

    def _get_noise_multiplier(self, current_epoch):
        # 激进一点：为了救 Crazing，我们可以让后期的噪声变得更小
        # 从 1.0 降到 0.1 (之前是 0.2)
        min_decay = 0.1 
        progress = current_epoch / self.total_epochs
        cosine_decay = 0.5 * (1 + math.cos(math.pi * progress))
        decay_factor = min_decay + (1 - min_decay) * cosine_decay
        return decay_factor

    def step(self, current_epoch):
        if current_epoch < 3: return

        decay_factor = self._get_noise_multiplier(current_epoch)

        # AGC
        grad_norms = []
        for p in self.model.parameters():
            if p.requires_grad and p.grad is not None:
                grad_norms.append(p.grad.norm(2).item())
        if not grad_norms: return
        current_median = np.median(grad_norms)
        clip_val = max(0.01, min(current_median * 1.5, 10.0))

        # 调试打印
        if not hasattr(self, '_logged_this_epoch') or self._logged_this_epoch != current_epoch:
            print(f"🚑 [Rescue DEBUG] Epoch {current_epoch} | AGC: {clip_val:.4f} | Decay: {decay_factor:.4f} | Strategy: {self.strategy}")
            self._logged_this_epoch = current_epoch

        # 策略计算
        current_device = next(self.model.parameters()).device
        
        # 收集需要梯度的参数
        param_list = [p for p in self.model.parameters() if p.requires_grad and p.grad is not None]
        names_list = [n for n, p in self.model.named_parameters() if p.requires_grad and p.grad is not None]
        
        factors = []
        for n in names_list:
            layer_factor = 1.0
            # 如果策略是 uniform，所有层都是 1.0，公平对待 Backbone
            if self.strategy == 'adaptive_smart':
                try:
                    layer_idx = int(n.split('.')[1])
                    if layer_idx in self.head_indices: layer_factor = 1.5
                    elif layer_idx in self.backbone_indices: layer_factor = 0.8
                except: pass
            factors.append(layer_factor)

        factors = torch.tensor(factors, device=current_device)
        # 归一化权重
        weights = factors / factors.sum() * len(factors) 
        
        idx = 0
        for p in param_list:
            layer_eps = self.epsilon * weights[idx]
            torch.nn.utils.clip_grad_norm_(p, clip_val)
            
            c = np.sqrt(2 * np.log(1.25 / 1e-5))
            base_sigma = c * clip_val / (layer_eps + 1e-8)
            final_sigma = base_sigma * decay_factor
            
            p.grad.add_(torch.randn_like(p.grad) * final_sigma)
            idx += 1

# ================= 2. 训练器 =================
class Trainer_Rescue(DetectionTrainer):
    def get_model(self, cfg=None, weights=None, verbose=True):
        model = super().get_model(cfg, weights, verbose)
        # ⚠️ 关键点：strategy='uniform'
        self.privacy_engine = PrivacyEngine_Annealing_Rescue(model, epsilon=10.0, strategy='uniform', epochs=30)
        return model
    
    def optimizer_step(self):
        self.scaler.unscale_(self.optimizer)
        self.privacy_engine.step(self.epoch)
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), 10.0)
        self.scaler.step(self.optimizer)
        self.scaler.update()
        self.optimizer.zero_grad()
        if self.ema: self.ema.update(self.model)

# ================= 3. 执行 Exp 18 =================
FULL_YAML = "/root/autodl-tmp/exp1/NEU_DET_YOLO/neu_det.yaml"
# 依然使用 GC10 预训练，因为这是唯一能提供纹理先验的来源
PRETRAINED_GC10 = "/root/autodl-tmp/exp1/result_exp1/13_GC10_Pretrain_Base/weights/best.pt"

print("🚀 开始 Exp 18: 龟裂拯救计划 (Texture Rescue)...")
print("✅ 策略: GC10预训练 + 解冻骨干 + 均匀退火 (不再牺牲Backbone)")

if os.path.exists(PRETRAINED_GC10):
    trainer_rescue = Trainer_Rescue(overrides={
        'model': PRETRAINED_GC10,
        'data': FULL_YAML,
        'epochs': 30,
        'batch': 16,
        'imgsz': 640,
        'project': 'result_exp1',
        'name': '18_Rescue_Crazing',
        'device': '0',
        'exist_ok': True,
        'freeze': 0  # 🔥 关键：完全解冻！让 Backbone 适应 Crazing
    })
    trainer_rescue.train()
else:
    print("❌ 没找到 Exp 13 的权重，无法执行拯救计划。")

🚀 开始 Exp 18: 龟裂拯救计划 (Texture Rescue)...
✅ 策略: GC10预训练 + 解冻骨干 + 均匀退火 (不再牺牲Backbone)
Ultralytics 8.3.248 🚀 Python-3.10.8 torch-2.1.2+cu118 CUDA:0 (NVIDIA GeForce RTX 4090, 24111MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/root/autodl-tmp/exp1/NEU_DET_YOLO/neu_det.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/root/autodl-tmp/exp1/result_exp1/13_GC10_Pretrain_Base/weights/best.pt, momentum=0.937, mosa

In [2]:
import os
from ultralytics import YOLO

# ================= 配置区域 =================
# 1. 你的 NEU-DET 数据配置
FULL_YAML = "/root/autodl-tmp/exp1/NEU_DET_YOLO/neu_det.yaml"

# 2. 你的 GC10 预训练权重 (必须和 Exp 18 用同一个！)
# 请确保这个路径是存在的
PRETRAINED_GC10 = "/root/autodl-tmp/exp1/result_exp1/13_GC10_Pretrain_Base/weights/best.pt"

print("🚀 开始 Exp 18.1对比实验: 无噪声纯迁移 (Clean Transfer Baseline)...")
print("🎯 目的: 测定模型在没有任何隐私束缚下的'性能天花板'，计算隐私代价 (Utility Loss)。")

if os.path.exists(PRETRAINED_GC10):
    # 直接加载模型，不需要自定义 Trainer
    model = YOLO(PRETRAINED_GC10)
    
    model.train(
        data=FULL_YAML,
        epochs=30,       # 保持轮数一致
        batch=16,        # 保持 batch 一致
        imgsz=640,
        project='result_exp1',
        name='18.1_Clean_Transfer_Baseline', # 实验名
        device='0',
        exist_ok=True,
        freeze=0,        # 保持和 Exp 18 一致，完全解冻
        plots=True       # 画出训练曲线
    )
else:
    print(f"❌ 错误: 找不到权重文件 {PRETRAINED_GC10}")

🚀 开始 Exp 18.1对比实验: 无噪声纯迁移 (Clean Transfer Baseline)...
🎯 目的: 测定模型在没有任何隐私束缚下的'性能天花板'，计算隐私代价 (Utility Loss)。
New https://pypi.org/project/ultralytics/8.3.252 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.248 🚀 Python-3.10.8 torch-2.1.2+cu118 CUDA:0 (NVIDIA GeForce RTX 4090, 24111MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/root/autodl-tmp/exp1/NEU_DET_YOLO/neu_det.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=3

In [3]:
import os
from ultralytics import YOLO
from ultralytics.models.yolo.detect import DetectionTrainer
import torch
import torch.nn as nn
import numpy as np

# ================= 1. 最原始的隐私引擎 (Basic DP) =================
# 没有 AGC，没有退火，没有 SA-LDP，单纯的 SGD-DP
class PrivacyEngine_Basic:
    def __init__(self, model, epsilon=10.0):
        self.model = model
        self.epsilon = epsilon
        # 固定裁剪阈值 (最笨的方法)
        self.max_grad_norm = 10.0 

    def step(self, current_epoch):
        # 即使是基础版，也给它一点 Warmup 吧，不然一开始就炸了也不公平
        if current_epoch < 1: return

        # 1. 全局裁剪
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.max_grad_norm)
        
        # 2. 加固定噪声
        # sigma = C * sqrt(2ln...) / epsilon
        c = np.sqrt(2 * np.log(1.25 / 1e-5))
        sigma = c * self.max_grad_norm / (self.epsilon + 1e-8)
        
        for p in self.model.parameters():
            if p.requires_grad and p.grad is not None:
                noise = torch.randn_like(p.grad) * sigma
                p.grad.add_(noise)

# ================= 2. 训练器 =================
class Trainer_Basic_Transfer(DetectionTrainer):
    def get_model(self, cfg=None, weights=None, verbose=True):
        model = super().get_model(cfg, weights, verbose)
        # 加载最原始的引擎
        self.privacy_engine = PrivacyEngine_Basic(model, epsilon=10.0)
        return model
    
    def optimizer_step(self):
        self.scaler.unscale_(self.optimizer)
        self.privacy_engine.step(self.epoch)
        self.scaler.step(self.optimizer)
        self.scaler.update()
        self.optimizer.zero_grad()
        if self.ema: self.ema.update(self.model)

# ================= 3. 执行 Exp 20 =================
FULL_YAML = "/root/autodl-tmp/exp1/NEU_DET_YOLO/neu_det.yaml"
PRETRAINED_GC10 = "/root/autodl-tmp/exp1/result_exp1/13_GC10_Pretrain_Base/weights/best.pt"

print("🚀 开始 Exp 18.2: 迁移 + 基础DP (Transfer + Basic DP)...")
print("🎯 目的: 证明提升不仅仅来自数据，算法优化(AGC+退火)也有巨大贡献。")

if os.path.exists(PRETRAINED_GC10):
    trainer = Trainer_Basic_Transfer(overrides={
        'model': PRETRAINED_GC10,
        'data': FULL_YAML,
        'epochs': 30,
        'batch': 16,
        'imgsz': 640,
        'project': 'result_exp1',
        'name': '18.2_Transfer_BasicDP_Ablation', # 消融实验
        'device': '0',
        'exist_ok': True,
        'freeze': 0
    })
    trainer.train()

🚀 开始 Exp 18.2: 迁移 + 基础DP (Transfer + Basic DP)...
🎯 目的: 证明提升不仅仅来自数据，算法优化(AGC+退火)也有巨大贡献。
Ultralytics 8.3.248 🚀 Python-3.10.8 torch-2.1.2+cu118 CUDA:0 (NVIDIA GeForce RTX 4090, 24111MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/root/autodl-tmp/exp1/NEU_DET_YOLO/neu_det.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/root/autodl-tmp/exp1/result_exp1/13_GC10_Pretrain_Base/weights/best.pt, momentum=0.937,

In [1]:
import os
from ultralytics import YOLO
from ultralytics.models.yolo.detect import DetectionTrainer
import torch
import torch.nn as nn
import numpy as np
import math

# ================= 1. 噪声退火引擎 (复用 Exp 16 逻辑) =================
class PrivacyEngine_Annealing_Rescue:
    def __init__(self, model, epsilon=10.0, strategy='uniform', epochs=30):
        self.model = model
        self.epsilon = epsilon
        self.strategy = strategy # 这里我们将传入 'uniform'
        self.total_epochs = epochs
        
        # 依然需要识别 Head，但如果是 uniform 策略，这个其实不影响计算
        self.head_indices = self._find_head_indices()
        self.backbone_indices = list(range(10))

    def _find_head_indices(self):
        indices = []
        for name, module in self.model.named_modules():
            if hasattr(module, '__class__') and 'Detect' in str(module.__class__):
                try:
                    idx = int(name.split('.')[1])
                    indices.append(idx)
                except: pass
        if not indices: indices = [23]
        return indices

    def _get_noise_multiplier(self, current_epoch):
        # 激进一点：为了救 Crazing，我们可以让后期的噪声变得更小
        # 从 1.0 降到 0.1 (之前是 0.2)
        min_decay = 0.1 
        progress = current_epoch / self.total_epochs
        cosine_decay = 0.5 * (1 + math.cos(math.pi * progress))
        decay_factor = min_decay + (1 - min_decay) * cosine_decay
        return decay_factor

    def step(self, current_epoch):
        if current_epoch < 3: return

        decay_factor = self._get_noise_multiplier(current_epoch)

        # AGC
        grad_norms = []
        for p in self.model.parameters():
            if p.requires_grad and p.grad is not None:
                grad_norms.append(p.grad.norm(2).item())
        if not grad_norms: return
        current_median = np.median(grad_norms)
        clip_val = max(0.01, min(current_median * 1.5, 10.0))

        # 调试打印
        if not hasattr(self, '_logged_this_epoch') or self._logged_this_epoch != current_epoch:
            print(f"🚑 [Rescue DEBUG] Epoch {current_epoch} | AGC: {clip_val:.4f} | Decay: {decay_factor:.4f} | Strategy: {self.strategy}")
            self._logged_this_epoch = current_epoch

        # 策略计算
        current_device = next(self.model.parameters()).device
        
        # 收集需要梯度的参数
        param_list = [p for p in self.model.parameters() if p.requires_grad and p.grad is not None]
        names_list = [n for n, p in self.model.named_parameters() if p.requires_grad and p.grad is not None]
        
        factors = []
        for n in names_list:
            layer_factor = 1.0
            # 如果策略是 uniform，所有层都是 1.0，公平对待 Backbone
            if self.strategy == 'adaptive_smart':
                try:
                    layer_idx = int(n.split('.')[1])
                    if layer_idx in self.head_indices: layer_factor = 1.5
                    elif layer_idx in self.backbone_indices: layer_factor = 0.8
                except: pass
            factors.append(layer_factor)

        factors = torch.tensor(factors, device=current_device)
        # 归一化权重
        weights = factors / factors.sum() * len(factors) 
        
        idx = 0
        for p in param_list:
            layer_eps = self.epsilon * weights[idx]
            torch.nn.utils.clip_grad_norm_(p, clip_val)
            
            c = np.sqrt(2 * np.log(1.25 / 1e-5))
            base_sigma = c * clip_val / (layer_eps + 1e-8)
            final_sigma = base_sigma * decay_factor
            
            p.grad.add_(torch.randn_like(p.grad) * final_sigma)
            idx += 1

# ================= 2. 训练器 =================
class Trainer_Rescue(DetectionTrainer):
    def get_model(self, cfg=None, weights=None, verbose=True):
        model = super().get_model(cfg, weights, verbose)
        # ⚠️ 关键点：strategy='uniform'
        self.privacy_engine = PrivacyEngine_Annealing_Rescue(model, epsilon=10.0, strategy='uniform', epochs=30)
        return model
    
    def optimizer_step(self):
        self.scaler.unscale_(self.optimizer)
        self.privacy_engine.step(self.epoch)
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), 10.0)
        self.scaler.step(self.optimizer)
        self.scaler.update()
        self.optimizer.zero_grad()
        if self.ema: self.ema.update(self.model)

# ================= 3. 执行 Exp 18 =================
FULL_YAML = "/root/autodl-tmp/exp1/NEU_DET_YOLO/neu_det.yaml"
# 依然使用 GC10 预训练，因为这是唯一能提供纹理先验的来源
#PRETRAINED_GC10 = "/root/autodl-tmp/exp1/result_exp1/13_GC10_Pretrain_Base/weights/best.pt"

#对比实验


print("🚀 开始 Exp 19: 对比实验")
print("✅ 策略: 无预训练 + 解冻骨干 + 均匀退火 (不再牺牲Backbone)")

trainer_rescue = Trainer_Rescue(overrides={
    'model': 'yolo11n.pt',
    'data': FULL_YAML,
    'epochs': 30,
    'batch': 16,
    'imgsz': 640,
    'project': 'result_exp1',
    'name': '19_No_GC-10',
    'device': '0',
    'exist_ok': True,
    'freeze': 0  # 🔥 关键：完全解冻！让 Backbone 适应 Crazing
})
trainer_rescue.train()

🚀 开始 Exp 19: 对比实验
✅ 策略: 无预训练 + 解冻骨干 + 均匀退火 (不再牺牲Backbone)
Ultralytics 8.3.248 🚀 Python-3.10.8 torch-2.1.2+cu118 CUDA:0 (NVIDIA GeForce RTX 4090, 24111MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/root/autodl-tmp/exp1/NEU_DET_YOLO/neu_det.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=19_No_GC-10, nbs=64, nms=False, opset=None, optimize=F

In [1]:
import warnings
warnings.filterwarnings('ignore')
import torch
import torch.nn as nn
import torch.optim as optim
import math
import numpy as np
import os
from ultralytics.models.yolo.detect import DetectionTrainer
from ultralytics.utils import colorstr

# ==========================================
# 1. 均衡版隐私引擎 (PrivacyEngine Balanced)
# ==========================================
class PrivacyEngine_Balanced_Rescue:
    def __init__(self, model, epsilon=10.0, strategy='uniform', epochs=50):
        self.model = model
        self.epsilon = epsilon
        self.strategy = strategy
        self.total_epochs = epochs
        # 记录一下是否打印过调试信息
        self._logged_this_epoch = -1

    def _get_noise_multiplier(self, current_epoch):
        # [修改点 1] 噪声底线提升到 0.2
        # 理由：0.1 会导致 MIA 攻击成功率飙升到 90%，0.2 是安全与精度的平衡点
        min_decay = 0.2 
        
        progress = current_epoch / self.total_epochs
        # 余弦退火策略
        cosine_decay = 0.5 * (1 + math.cos(math.pi * progress))
        decay_factor = min_decay + (1 - min_decay) * cosine_decay
        return decay_factor

    def step(self, current_epoch):
        # 前 3 轮预热不加噪，保证训练初期稳定
        if current_epoch < 3: return

        # 计算当前噪声衰减系数
        decay_factor = self._get_noise_multiplier(current_epoch)

        # === 自适应梯度裁剪 (AGC) ===
        grad_norms = []
        for p in self.model.parameters():
            if p.requires_grad and p.grad is not None:
                grad_norms.append(p.grad.norm(2).item())
        
        if not grad_norms: return # 避免空梯度报错

        current_median = np.median(grad_norms)

        # [修改点 2] 裁剪阈值放宽到 2.5 倍中位数
        # 理由：之前的 1.5 把 Crazing (难样本) 的梯度切没了。
        # 2.5 既能让大梯度通过，又不至于像 4.0 那样引入过量噪声。
        clip_val = max(0.01, min(current_median * 2.5, 5.0))

        # 打印调试信息 (每个 Epoch 只打一次)
        if self._logged_this_epoch != current_epoch:
            print(f"\n🚑 [Balanced Rescue] Epoch {current_epoch}/{self.total_epochs}")
            print(f"   - AGC Threshold (Clip): {clip_val:.4f} (Median * 2.5)")
            print(f"   - Noise Factor (Decay): {decay_factor:.4f} (Min 0.2)")
            self._logged_this_epoch = current_epoch

        # === 注入噪声 ===
        current_device = next(self.model.parameters()).device
        
        # 收集参数
        param_list = [p for p in self.model.parameters() if p.requires_grad and p.grad is not None]
        
        # 计算每一层的噪声强度
        # Uniform 策略：所有层一视同仁
        weights = torch.ones(len(param_list), device=current_device)
        
        idx = 0
        for p in param_list:
            # 1. 裁剪梯度
            torch.nn.utils.clip_grad_norm_(p, clip_val)
            
            # 2. 计算噪声标准差 Sigma
            # 公式: sigma = (c * clip) / epsilon
            layer_eps = self.epsilon * weights[idx] # uniform下权重为1
            c = np.sqrt(2 * np.log(1.25 / 1e-5))
            base_sigma = c * clip_val / (layer_eps + 1e-8)
            
            # 应用衰减
            final_sigma = base_sigma * decay_factor
            
            # 3. 加噪
            noise = torch.randn_like(p.grad) * final_sigma
            p.grad.add_(noise)
            idx += 1

# ==========================================
# 2. 定制训练器
# ==========================================
class Trainer_Rescue(DetectionTrainer):
    def get_model(self, cfg=None, weights=None, verbose=True):
        model = super().get_model(cfg, weights, verbose)
        # [修改点 3] 传入 50 epochs
        self.privacy_engine = PrivacyEngine_Balanced_Rescue(
            model, 
            epsilon=10.0, 
            strategy='uniform', 
            epochs=50 
        )
        return model
    
    def optimizer_step(self):
        self.scaler.unscale_(self.optimizer)
        # 调用隐私引擎处理梯度
        self.privacy_engine.step(self.epoch)
        # 全局裁剪防止梯度爆炸 (兜底策略)
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), 10.0)
        self.scaler.step(self.optimizer)
        self.scaler.update()
        self.optimizer.zero_grad()
        if self.ema: self.ema.update(self.model)

# ==========================================
# 3. 运行配置 (Exp 19)
# ==========================================
if __name__ == "__main__":
    # 请确认路径是否正确
    FULL_YAML = "/root/autodl-tmp/exp1/NEU_DET_YOLO/neu_det.yaml"
    # 继续使用 GC10 预训练，利用其纹理先验
    PRETRAINED_GC10 = "/root/autodl-tmp/exp1/result_exp1/13_GC10_Pretrain_Base/weights/best.pt"

    print("🚀 开始 Exp 20: 均衡救援计划 (Balanced Rescue)...")
    print("📌 配置要点:")
    print("   1. Clip: Median * 2.5 (救 Crazing)")
    print("   2. Noise Min: 0.2 (防 MIA)")
    print("   3. Epochs: 50 (时间换空间)")

    if os.path.exists(PRETRAINED_GC10):
        trainer = Trainer_Rescue(overrides={
            'model': PRETRAINED_GC10,
            'data': FULL_YAML,
            'epochs': 50,       # 延长训练时间
            'batch': 16,
            'imgsz': 640,
            'project': 'result_exp1',
            'name': '20_Rescue_Balanced', # 新的实验名称
            'device': '0',
            'exist_ok': True,
            'freeze': 0,        # 完全解冻，让 Backbone 适应
            'optimizer': 'SGD', # 推荐用 SGD，对噪声更鲁棒
            'lr0': 0.01,        # 初始学习率
            'lrf': 0.01         # 最终学习率
        })
        trainer.train()
    else:
        print(f"❌ 找不到预训练权重: {PRETRAINED_GC10}")

🚀 开始 Exp 20: 均衡救援计划 (Balanced Rescue)...
📌 配置要点:
   1. Clip: Median * 2.5 (救 Crazing)
   2. Noise Min: 0.2 (防 MIA)
   3. Epochs: 50 (时间换空间)
Ultralytics 8.3.248 🚀 Python-3.10.8 torch-2.1.2+cu118 CUDA:0 (NVIDIA GeForce RTX 4090, 24111MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/root/autodl-tmp/exp1/NEU_DET_YOLO/neu_det.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/root/autodl-tmp/exp1/result_exp1/13